# Volume 7. Fiber Circuit Gating

Question: how can a notebook show control over which projection channels run?

`FiberCircuit` is a small scheduling helper. It does not prove biological gating, but it gives us an inspectable control protocol.

In [ ]:
import pandas as pd

from neural_assemblies.assembly_calculus import FiberCircuit
from neural_assemblies.core.brain import Brain


In [ ]:
N = 1_500
K = 30
brain = Brain(p=0.05, save_winners=True, seed=17, engine="numpy_sparse")
brain.add_stimulus("cue", K)
for area in ["SENSORY", "WORKING", "ACTION"]:
    brain.add_area(area, N, K, beta=0.06)

circuit = FiberCircuit(brain)
circuit.add_stim("cue", "SENSORY", active=True)
circuit.add("SENSORY", "WORKING", active=False)
circuit.add("WORKING", "ACTION", active=False)


In [ ]:
def winner_counts():
    return {area: len(brain.areas[area].winners) for area in ["SENSORY", "WORKING", "ACTION"]}

rows = []

circuit.step()
rows.append({"step": "cue sensory", **winner_counts(), "active_area_fibers": circuit.active_area_projections()})

circuit.inhibit("cue", "SENSORY")
circuit.disinhibit("SENSORY", "WORKING")
circuit.step()
rows.append({"step": "sensory working", **winner_counts(), "active_area_fibers": circuit.active_area_projections()})

circuit.inhibit("SENSORY", "WORKING")
circuit.disinhibit("WORKING", "ACTION")
circuit.step()
rows.append({"step": "working action", **winner_counts(), "active_area_fibers": circuit.active_area_projections()})

pd.DataFrame(rows)


Try next: add a recurrent `WORKING -> WORKING` fiber and compare whether the working state persists after the stimulus fiber is off.